# Predictive Anayltics: Support Vector Machines with Regression for Census Tract

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [55]:
## could not allocate memory for 1h check

In [56]:
%load_ext cuml.accel
from run_config import PATHS

The cuml.accel extension is already loaded. To reload it, use:
  %reload_ext cuml.accel


In [57]:
import os
os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

import cuml
print(cuml.__version__)

26.06.00


In [58]:
GRID_SAMPLE = 70_000 # if validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "24H" # options: 1H, 4H, 24H

In [59]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [60]:
import pandas as pd
import numpy as np
import datetime
#from sklearn.svm import SVR 
#from sklearn.svm import LinearSVR
from cuml import SVR
from cuml import LinearSVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
# explicitly require this experimental feature
from sklearn.experimental import enable_halving_search_cv # noqa
# now you can import normally from model_selection
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
from joblib import Memory
import h3

#from shapely import wkt

#import geopandas as gpd
#from shapely.geometry import Polygon
#from srai.neighbourhoods import H3Neighbourhood
#from srai.loaders import OSMOnlineLoader
#from srai.joiners import IntersectionJoiner
#from srai.h3 import ring_buffer_h3_regions_gdf
#from srai.embedders import Hex2VecEmbedder

#import networkx as nx
#from libpysal.weights import Queen
#from node2vec import Node2Vec

## Preparations

In [61]:
INPUT = PATHS.train_test_dir

In [62]:
SPATIAL_UNIT = "CENSUS_TRACTS"

# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [63]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [64]:
# gpu kernel does not work with higher count, runs into non-sensical numbers
if len(train_df) > 70_000: 
    train_df = train_df.sample(n=70_000, random_state=40)

In [65]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [66]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [67]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
107921,2025-02-25,2,2,0,0.500000,8.660254e-01,0.781831,0.623490,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
260959,2025-06-25,6,3,0,0.500000,-8.660254e-01,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
192816,2026-03-07,3,6,0,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
160050,2026-02-14,2,6,0,0.500000,8.660254e-01,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,371.52,10.927059,6.32,16.5,Mobile
258156,2025-10-01,10,3,0,-1.000000,-1.836970e-16,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119140,2025-01-29,1,3,0,0.000000,1.000000e+00,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
277208,2026-04-20,4,1,0,1.000000,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
77731,2025-06-03,6,2,0,0.500000,-8.660254e-01,0.781831,0.623490,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
153255,2025-05-22,5,4,0,0.866025,-5.000000e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips


In [68]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: census_tract")
       
    # load census tract
    census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
    census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

    tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
    tract_centroids.columns = ["lat", "lon"]

    for df in (train_df, val_df, test_df):
        df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
        df["lat"] = df["census_tract"].map(tract_centroids["lat"])
        df["lon"] = df["census_tract"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing = df["lat"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
        n_missing = df["lon"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

Encoding: latlong and Unit: census_tract


In [69]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,lat,lon
107921,2025-02-25,2,2,0,0.500000,8.660254e-01,0.781831,0.623490,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,42.001917,-87.674584
260959,2025-06-25,6,3,0,0.500000,-8.660254e-01,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.855441,-87.682019
192816,2026-03-07,3,6,0,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.776911,-87.603257
160050,2026-02-14,2,6,0,0.500000,8.660254e-01,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,371.52,10.927059,6.32,16.5,Mobile,41.869600,-87.641007
258156,2025-10-01,10,3,0,-1.000000,-1.836970e-16,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.761702,-87.629872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119140,2025-01-29,1,3,0,0.000000,1.000000e+00,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.723655,-87.610825
277208,2026-04-20,4,1,0,1.000000,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.942169,-87.771496
77731,2025-06-03,6,2,0,0.500000,-8.660254e-01,0.781831,0.623490,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.761435,-87.656447
153255,2025-05-22,5,4,0,0.866025,-5.000000e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.00,0.000000,0.00,0.0,No trips,41.667800,-87.633393


In [70]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else:
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

Create y

In [71]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Grid Search

In [72]:
model = SVR()

In [73]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300],
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300],
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.07120059545582968 best params: {'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
rbf_sigmoid best score: 0.11301351545235871 best params: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}
poly best score: 0.035095473906050834 best params: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 100, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}
Overall best: rbf_sigmoid {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}


In [74]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.01}
Best CV score: 0.11301351545235871


### Train Model

In [75]:
best_model = grid_search.best_estimator_

In [76]:
# Train SVR 

best_model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(memo...svm', SVR())])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: str or callable, default='rbf'Kernel map to be approximated. A callable should accept two argumentsand the keyword arguments passed to this object as `kernel_params`, andshould return a floating point number.",'rbf'
,"gamma gamma: float, default=NoneGamma parameter for the RBF, laplacian, polynomial, exponential chi2and sigmoid kernels. Interpretation of the default value is left tothe kernel; see the documentation for sklearn.metrics.pairwise.Ignored by other kernels.",0.01
,"coef0 coef0: float, default=NoneZero coefficient for polynomial and sigmoid kernels.Ignored by other kernels.",None


In [77]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [78]:
y_pred

array([9.26901484, 0.01700245, 3.33807498, ..., 6.86251899, 0.07560685,
       6.04144247], shape=(64094,))

In [79]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 9.028422476244065
MSE: 1998.2733604831044
RMSE: 44.702050965063165
R2 Score: 0.7035828513097031


In [80]:
# save model
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/svm/grid_CENSUS_TRACTS_24H_svr.joblib']